In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Importing modules

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, module='pandas')
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import Dataset, Reader, SVD, accuracy ,NMF, KNNBasic
from surprise.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from surprise.model_selection import GridSearchCV
import math
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
import math

In [ ]:
!pip install kneed

# Loading data

This is a tab separated list of

user id | item id | rating | timestamp. 

The time stamps are unix seconds will convert it 

In [ ]:
data_df = pd.read_csv("/kaggle/input/movie-lens/ml-100k/u.data",sep='\t', names=["user_id",'movie_id','rating','timestamp'])
data_df['timestamp'] = pd.to_datetime(data_df['timestamp'], unit='s')
data_df.head(10)

In [ ]:
# extra specific content will se if it is needed
# data_df['year'] = data_df['timestamp'].dt.year
# data_df['month'] = data_df['timestamp'].dt.month
# data_df['day_of_week'] = data_df['timestamp'].dt.dayofweek
# data_df['hour'] = data_df['timestamp'].dt.hour

In [ ]:
data_df.info()

No null values are present

In [ ]:
data_df.describe()

In [ ]:
user_df = pd.read_csv("/kaggle/input/movie-lens/ml-100k/u.user",sep='\|',names=["user_id" , "age", "gender" , "occupation" , "zip_code"])
user_df.head(10)

In [ ]:
user_df.info()

In [ ]:
user_df.describe()

The user age ranges from 7 years to 73 years with median 31 years 

In [ ]:
user_df.nunique()

In [ ]:
item_df = pd.read_csv("/kaggle/input/movie-lens/ml-100k/u.item", 
                      sep='|', encoding='latin-1',
                      header=None, 
                      names=['movie_id' , 'movie_title' , 'release_date' , 'video_release date' , 'IMDb_URL' , 'unknown'
                             , 'Action' , 'Adventure' , 'Animation' , "Children's" , 'Comedy' , 'Crime' , 'Documentary'
                             , 'Drama', 'Fantasy' , 'Film-Noir' , 'Horror' , 'Musical' , 'Mystery' , 'Romance' ,
                             'Sci-Fi' , 'Thriller' , 'War' , 'Western' ]) 
item_df.head()

this is a tab separatedlist of
              **movie id | movie title | release date | video release date |
              IMDb URL | unknown | Action | Adventure | Animation |
              Children's | Comedy | Crime | Documentary | Drama | Fantasy |
              Film-Noir | Horror | Musical | Mystery | Romance | Sci-Fi |
              Thriller | War | Western |**
              The last 19 fields are the genres, a 1 indicates the movie
              is of that genre, a 0 indicates it is not; movies can be in
              several genres at once.
              The movie ids are the ones used in the u.data data set.

In [ ]:
item_df.info()

imdb url has very few null and video release date is null for all so we will drop video release date and handle 

For item_df, column names like "Children's" might cause syntax issues later if you refer to them in code — not a big problem but worth being aware of.

You could cast some int64 columns (like user_id, movie_id, genre flags) to int32 to reduce memory usage, though this isn’t critical unless scaling up.

💡 **Business & ML Intuition: Why Clean Data Matters Here**
In recommendation systems, dirty data (like missing URLs or detached metadata) costs business value. If a model recommends a movie but the frontend cannot fetch its poster or IMDb link because of a null value, the user experience breaks. Always ensure data integrity before feeding it to collaborative filtering, as silent failures here lead to dead UI components.

# Data cleaning

In [ ]:
item_df.drop('video_release date', axis= 1,inplace=True)
item_df = item_df[item_df['movie_id'] != 267]
item_df['IMDb_URL'].fillna('imputed_url', inplace=True)

# Data merging

In [ ]:
merged_df = pd.merge(data_df, item_df, on='movie_id',how = "inner")
df = pd.merge(merged_df, user_df, on='user_id')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

# EDA

In [ ]:
df.head()

In [ ]:
# Compute global summary (run once)
num_users = df['user_id'].nunique()
num_movies = df['movie_id'].nunique()
num_ratings = len(df)
sparcity = 1 - (num_ratings / (num_users * num_movies))  # fraction of cells that are empty

summary = {
    "num_users": int(num_users),
    "num_movies": int(num_movies),
    "num_ratings": int(num_ratings),
    "sparsity": float(sparcity),
    "avg_rating": float(df['rating'].mean()),
    "median_rating": float(df['rating'].median()),
    "std_rating": float(df['rating'].std())
}

In [ ]:
# Save summary
import json
with open('/kaggle/working/insights_summary.json','w') as f:
    json.dump(summary, f, indent=2)

import pandas as pd
pd.DataFrame([summary]).to_csv('/kaggle/working/insights_summary.csv', index=False)

summary

In [ ]:
popular_movies_df = (
    ratings_df_filtered.groupby('movie_id')
    .agg(num_ratings=('rating', 'count'), avg_rating=('rating', 'mean'))
    .reset_index()
    .merge(item_df[['movie_id', 'movie_title']], on='movie_id', how='left')
    .sort_values(['num_ratings', 'avg_rating'], ascending=False)
)

### Rating distribution (histogram)

In [ ]:
counts, bins, patches = plt.hist(df['rating'], bins=[0.5,1.5,2.5,3.5,4.5,5.5], rwidth=0.8)
plt.xticks([1,2,3,4,5])
plt.xlabel('Rating')
plt.ylabel('Number of Ratings')
plt.title('Rating Distribution')

# annotate counts
for c, b in zip(counts, bins[:-1]):
    plt.text(b+0.45, c+200, int(c), ha='center')

# mean & median
m = df['rating'].mean()
med = df['rating'].median()
plt.axvline(m, color='red', linestyle='--', label=f"mean={m:.2f}")
plt.axvline(med, color='green', linestyle=':', label=f"median={med:.2f}")
plt.legend()
plt.tight_layout()
plt.show()


### Rating distribution

This histogram shows the counts of ratings at each star value. The distribution peaks around 3–4 stars, indicating that users tend to give moderately positive ratings.  
Observations:
- Rating counts per value (from the dataset): 1 → 6,109; 2 → 11,370; 3 → 27,142; 4 → 34,170; 5 → 21,200.
- Average rating ≈ 3.53, suggesting a positive bias.
Implications:
- Most models should expect the central tendency around 3–4.
- When evaluating, consider thresholding “relevant” at ≥4.0 for Precision@K.


In [ ]:
hist_counts = dict(zip([1,2,3,4,5], [6109,11370,27142,34170,21200]))
import pandas as pd
pd.Series(hist_counts, name='count').to_csv('/kaggle/working/rating_counts.csv')

In [ ]:
rating_stats = df.groupby('movie_id')['rating'].agg(['mean','count']).reset_index().rename(columns={'mean':'avg_rating','count':'num_rating'})
corr = rating_stats['avg_rating'].corr(rating_stats['num_rating'])
plt.figure(figsize=(10,6))
sns.scatterplot(data=rating_stats, x='num_rating', y='avg_rating', alpha=0.5)
plt.title(f'Average Rating vs Number of Ratings (corr={corr:.3f})')
plt.xlabel('Number of Ratings')
plt.ylabel('Average Rating')
plt.grid(True)
plt.tight_layout()
plt.show()

rating_stats.to_csv('/kaggle/working/movie_rating_stats.csv', index=False)


### Average rating vs. number of ratings

Scatter plot of each movie's average rating vs. its number of ratings. Observations:
- Many low-count movies have high variance in avg rating (noisy).
- Movies with a larger number of ratings converge toward ~3.5–4.0.
Implication:
- Use minimum vote thresholds (or Bayesian average) to reduce noise from low-count movies.
- The plot justifies the use of the weighted/popularity score.



In [ ]:
rating_stats = df.groupby('movie_id')['rating'].agg(['mean','count'])
rating_stats.columns = ['avg_rating','num_rating']
popular_movies = rating_stats[rating_stats['num_rating'] >= 50]
top_10_avg = popular_movies['avg_rating'].sort_values(ascending=False).head(10)

In [ ]:
print(top_10_avg)

In [ ]:
sci_fi_ratings = df[df['Sci-Fi'] == 1]

avg_scifi_rating_by_occupation = sci_fi_ratings.groupby('occupation')['rating'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
avg_scifi_rating_by_occupation.plot(kind='bar')
plt.title('Average Sci-Fi Movie Rating by User Occupation')
plt.xlabel('Occupation')
plt.ylabel('Average Rating')
plt.xticks(rotation=45, ha='right')
plt.show()

### How many movies each user rated

In [ ]:
user_rating_count = df.groupby('user_id')['rating'].count().nlargest(10)
plt.figure(figsize=(10,6))
ax = sns.barplot(x=[f"#{i+1}" for i in range(len(user_rating_count))], y=user_rating_count.values)
for p, v in zip(ax.patches, user_rating_count.values):
    ax.annotate(int(v), (p.get_x()+p.get_width()/2., p.get_height()), ha='center', va='center', xytext=(0,6), textcoords='offset points')
plt.xlabel('User Rank')
plt.ylabel('Number of Ratings')
plt.title('Top 10 Most Active Users')
plt.tight_layout()
plt.show()

user_rating_count.reset_index().to_csv('/kaggle/working/top10_active_users.csv', index=False)


### Most active users

This chart shows the users who rated the most movies. The top user contributed ~737 ratings — heavy users can skew collaborative filtering models if not handled.



⚠️ **Common Mistake: Trusting Raw Averages**
A movie with 1 rating of 5.0 is technically a '5-star' movie, but recommending it is extremely risky. Business Logic dictates that we need *confidence* in a rating. The Bayesian Average pulls items with very few ratings towards the global mean, preventing obscure, low-interaction items from hijacking the top-tier recommendation slots.

### (Baysian average formula): Weighted Popularity = (v / (v + m)) * R + (m / (v + m)) * C
### Where:
### R = avg rating of the movie
### v = num votes for the movie
### m = min votes required 
### C = mean of all movie ratings

If a movie has very few ratings (low v), we don’t fully trust its average rating R, so we pull it closer to the global average C.

In [ ]:
def calculate_popularity_score(df, m_quantile=0.25):
    rating_stats = df.groupby('movie_id')['rating'].agg(['mean','count']).reset_index()
    rating_stats.columns = ['movie_id','avg_rating','num_rating']

# global mean weighted by counts
    C = (rating_stats['avg_rating'] * rating_stats['num_rating']).sum() / rating_stats['num_rating'].sum()
    m = int(rating_stats['num_rating'].quantile(m_quantile))
    m = max(1, m)
    rating_stats['popularity_score'] = (
        (rating_stats['num_rating'] / (rating_stats['num_rating'] + m)) * rating_stats['avg_rating'] +
        (m / (rating_stats['num_rating'] + m)) * C
    )
    return rating_stats, C, m

In [ ]:
pop_df, global_mean, m_threshold = calculate_popularity_score(df, m_quantile=0.25)

plt.figure(figsize=(10,6))
sns.scatterplot(data=pop_df, x='num_rating', y='popularity_score', alpha=0.6)
plt.title('Popularity Score vs Number of Ratings')
plt.xlabel('Number of Ratings')
plt.ylabel('popularity_score')
plt.grid(True)
plt.tight_layout()
plt.show()

pop_df.to_csv('/kaggle/working/popularity_scores.csv', index=False)
print("Global mean:", global_mean, "m threshold:", m_threshold)

### Popularity score (Bayesian weighted) vs number of ratings

We compute a Bayesian-weighted score to prevent low-sample movies from ranking too high. The scatter shows that:
- After applying the weighted score, low-count movies are pulled towards the global mean.
- High-count high-rated titles remain high (more reliable).
This justifies using hybrid ranking: CF_score * α + popularity_score * (1-α).


### Top-10 by weighted popularity

In [ ]:
movie_titles_map = df.drop_duplicates(subset='movie_id').set_index('movie_id')['movie_title']
top_popular_with_titles = pop_df.sort_values('popularity_score', ascending=False).head(10)
top_popular_with_titles['movie_title'] = top_popular_with_titles['movie_id'].map(movie_titles_map)

# Save full details to CSV
top_popular_with_titles[['movie_id', 'movie_title', 'num_rating', 'avg_rating', 'popularity_score']] \
    .to_csv('/kaggle/working/top_popular_movies.csv', index=False)

# Display reduced version
top_popular_with_titles[['movie_id', 'movie_title', 'popularity_score']].reset_index(drop=True)


### Scatter Plot: Average Rating vs. Number of Ratings
shows why we need a smarter popularity score.

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=rating_stats, x='num_rating', y='avg_rating', alpha=0.5)
plt.title("Average Rating vs Number of Ratings")
plt.xlabel("Number of Ratings")
plt.ylabel("Average Rating")
plt.grid(True)
plt.show()

As we can see the left side is more clustered means very low number of ratings have high average ratings

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=pop_df, x='num_rating', y='popularity_score', alpha=0.6, color='green')
plt.title("Popularity Score vs Number of Ratings")
plt.xlabel("Number of Ratings")
plt.ylabel("popularity_score")
plt.grid(True)
plt.show()

few ratings get pulled toward the average.
Movies with many ratings dominate only if they also have high ratings.

In [ ]:
# Gender mean rating
gender_avg = df.groupby('gender')['rating'].mean().reset_index()
gender_avg.to_csv('/kaggle/working/gender_avg_rating.csv', index=False)

# Occupation counts plot and save
occ_counts = user_df['occupation'].value_counts().reset_index().rename(columns={'index':'occupation','occupation':'count'})
occ_counts.to_csv('/kaggle/working/occupation_counts.csv', index=False)

### Gender & occupation insights

- Mean rating by gender is nearly identical (F: 3.5315 vs M: 3.5293) — little evidence of strong gender-based rating bias.
- Occupation distribution: students are the single largest occupation group followed by 'other', 'educator', and 'administrator'. 
Implications:
- Occupation features could be useful for demographic-aware hybrid models, but gender alone appears not strongly predictive of rating magnitude.


### Sparsity & matrix size

- Users: {num_users}, Movies: {num_movies}.
- Ratings: {num_ratings}.
- Matrix sparsity ≈ {sparcity:.3f} (only ~{(1-sparcity)*100:.2f}% of user–item pairs have ratings).
Implication: Collaborative Filtering must handle extreme sparsity — consider matrix factorization (SVD) or hybrid approaches.


In [ ]:
# Consolidate to a small report CSV/JSON
insights = {
    'summary': summary,
    'top_popular': top_popular_with_titles[['movie_id','movie_title','popularity_score']].to_dict(orient='records'),
    'gender_avg': gender_avg.to_dict(orient='records'),
    'occupation_counts': occ_counts.to_dict(orient='records')
}
with open('/kaggle/working/full_insights.json','w') as f:
    json.dump(insights, f, indent=2)


## User Item Matrix

In [ ]:
user_item_matrix = df.pivot_table(index='user_id', columns='movie_id', values='rating')
user_item_matrix.head()

In [ ]:
# Density 
density  = user_item_matrix.count().sum()/user_item_matrix.size

6.3% of the matrix has actual ratings

In [ ]:
sparsity = 1-density
sparsity


###  If sparsity is high like more than 90% , matrix factorization makes more sense than content-based filtering.
### 93.7% of the matrix is empty

💡 **Business Context: The Cost of Sparsity**
A sparsity of >93% is typical in industry (most users haven't seen most movies). However, this is a massive challenge for memory-based models (KNN). If users have very few co-ratings, we can't calculate similarity. This is the main business driver for transitioning from basic CF to Matrix Factorization techniques (like SVD) which can infer relationships without direct co-ratings.

### Distribution of Ratings per User

In [ ]:
user_rating_count = user_item_matrix.count(axis=1)
plt.hist(user_rating_count, bins=50)
plt.title("Number of Ratings per User")
plt.xlabel("Ratings Count")
plt.ylabel("Number of Users")
plt.grid(True)
plt.show()


### Distribution of Ratings per Movie

In [ ]:
movie_rating_count = user_item_matrix.count()
plt.hist(movie_rating_count, bins=30)
plt.title("Number of Ratings per Movie")
plt.xlabel("Ratings Count")
plt.ylabel("Number of Movies")
plt.grid(True)
plt.show()

### users rated more than 100 movies.

In [ ]:
userWith100_Plus_movieRating = user_rating_count[user_rating_count >= 100]

In [ ]:
userWith25_minus_movieRating = user_rating_count[user_rating_count < 25]

In [ ]:
top_5 = (df.groupby(['movie_id', 'movie_title'])['rating'].agg(['mean', 'count'])
         .query('count >= 100')
         .sort_values('mean', ascending=False)
         .head(5))

print(top_5)

## Genre

### which genres appear most frequently in the dataset

In [ ]:
genres = ['unknown' ,'Action' , 'Adventure' , 'Animation' , "Children's" , 'Comedy' , 'Crime' , 'Documentary', 'Drama', 'Fantasy' , 
         'Film-Noir' , 'Horror' , 'Musical' , 'Mystery' , 'Romance' ,'Sci-Fi' , 'Thriller' , 'War' , 'Western' ]

Popular genres will be amplified in df because popular movies get rated more often!

In [ ]:
total_movies = len(item_df)
genre_counts = item_df[genres].sum().sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(range(len(genre_counts)), genre_counts.values)
plt.xticks(range(len(genre_counts)), genre_counts.index, rotation=45, ha='right')
plt.xlabel('Genres')
plt.ylabel('Number of Movies')
plt.title('Most Frequent Genres in MovieLens Dataset')
plt.tight_layout()
plt.show()

In [ ]:
genre_percentage = (genre_counts / total_movies*100 ).sort_values(ascending=False)
for genre , percentage in genre_percentage.items():
    count = genre_counts[genre]
    print(f"{genre}: {count} movies ({percentage:.1f}%)")

### Any very active or very inactive users?

In [ ]:
user_activity = df.groupby('user_id')['rating'].count()

very_active_threshold = user_activity.quantile(0.95)
very_active_user_ids = user_activity[user_activity >= very_active_threshold].index

very_inactive_threshold = user_activity.quantile(0.05) 
very_inactive_user_ids = user_activity[user_activity <= very_inactive_threshold].index

User × Movie Matrix (user_id as rows, movie_id or movie_title as columns, and rating as values)

In [ ]:
print(very_active_user_ids)
print(very_inactive_user_ids)

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
from kneed import KneeLocator
user_rating_counts = df.groupby('user_id').size().sort_values(ascending=False)

At first, the curve will drop slowly… then suddenly bend sharply (like an elbow).
That “bend” is the knee point — the place where keeping stricter thresholds will remove too many users/movies.

The knee point (also called elbow point) is the “sweet spot” where increasing a threshold starts giving diminishing returns.

💡 **Business Intuition: Trimming the Fat**
Why filter using the knee point? 
1. **Compute Cost:** Calculating user-user similarities on users who rated 2 movies is computationally expensive and yields noisy data.
2. **Model Quality:** Training on sparse 'casual' users degrades the quality of latent representations. By cutting at the elbow, we keep the most informative data while massively reducing training time and server costs.

groupby('user_id').size() → counts ratings per user

Sort values descending so most active users are first

KneeLocator scans the curve and finds where slope changes most

The threshold = number of ratings at that knee point

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1, len(user_rating_counts)+1), user_rating_counts.values)
plt.xlabel('User Rank')
plt.ylabel('Number of Ratings')
plt.title('Ratings per User')
plt.show()

In [ ]:
knee_user = KneeLocator(
    range(1, len(user_rating_counts)+1),
    user_rating_counts.values,
    curve='convex',
    direction='decreasing'
)

print("Knee point (Users):", knee_user.knee)
print("Min ratings per user threshold:", user_rating_counts.values[knee_user.knee-1])

In [ ]:
movie_rating_counts = df.groupby('movie_id').size().sort_values(ascending=False)
plt.figure(figsize=(8,5))
plt.plot(range(1, len(movie_rating_counts)+1), movie_rating_counts.values)
plt.xlabel('Movie Rank')
plt.ylabel('Number of Ratings')
plt.title('Ratings per Movie')
plt.show()

In [ ]:
knee_movie = KneeLocator(
    range(1, len(movie_rating_counts)+1),
    movie_rating_counts.values,
    curve='convex',
    direction='decreasing'
)

print("Knee point (Movies):", knee_movie.knee)
print("Min ratings per movie threshold:", movie_rating_counts.values[knee_movie.knee-1])

# Memory based colaborative filtering

 The cosine_similarity function works by comparing rows, so we must first transpose (.T) the matrix.

 Item-Item similarity is often preferred in practice as item attributes are more stable than user preferences, and it scales better when there are more users than items.

 #before consine similarity we need to handle null values in user item matrix. filling with 0 makes the non rated as 0 or neutral in other terms, might distort similarity since sparsity is high. We might later test centering ratings by subtracting each item's mean before similarity calculation.


Normalization for better accuracy.

Co-rating weighting.

Cold start handling.

### Cosine Similarity vs. Pearson Correlation: A Detailed Comparison

When building memory-based collaborative filters, choosing the right similarity metric is crucial as it defines what "similar taste" means. Below is a comparison of the two most common methods.

### **In-Depth: Cosine Similarity**

📐 **Cosine similarity** focuses on the orientation, not the magnitude, of the rating vectors. A high score (close to 1) means the vectors point in a similar direction.

* **Pro:** It's a straightforward and computationally efficient way to find users or items with similar tastes.
* **Con:** It can be misleading when users have different rating scales (e.g., a "harsh" rater vs. a "generous" rater).

> **Example:**
> * **User A (Generous):** Rates Movie1 `4` and Movie2 `5`.
> * **User B (Strict):** Rates Movie1 `2` and Movie2 `3`.
>
> Even though their raw ratings are far apart, the *pattern* is similar (both liked Movie2 more than Movie1). Cosine similarity will capture this directional similarity but doesn't account for the fact that User A is generally a more positive rater.

---

### **In-Depth: Pearson Correlation**

🎯 **Pearson correlation** first normalizes the data by subtracting the user's average rating from each of their individual ratings. This is called **mean-centering**. After centering, it essentially calculates the cosine similarity on the new, centered vectors.

* **Pro:** By removing the average rating bias, it can more accurately identify users who have similar *tastes* regardless of whether they are generally generous or strict with their scores.
* **Con:** It can be undefined for users who have rated all items with the exact same score (as there is no variance).

---

## Item-Item collaborative filtering

💡 **Industry Standard: Why Item-Item Wins in Production**
While User-User CF makes intuitive sense ('find people like me'), Item-Item is overwhelmingly preferred in large-scale production systems (like Amazon's early engine). 
- **Stability:** Item attributes and relationships (Star Wars vs. Star Trek) change very slowly. User tastes can pivot daily.
- **Compute & Caching:** We typically have way more users than items. It's much cheaper to compute an $I \times I$ matrix and cache it offline than to constantly compute a $U \times U$ matrix in real-time.

In [ ]:
def prepare_item_similarity(matrix, method='cosine', min_support=15):
    if method == 'cosine':
        centered_matrix = matrix.sub(matrix.mean(axis=0), axis=1).fillna(0)
        similarity_matrix = pd.DataFrame(cosine_similarity(centered_matrix.T), 
                                         index=matrix.columns, 
                                         columns=matrix.columns)
    elif method == 'pearson':
        # Calculate Pearson correlation as before
        similarity_matrix = matrix.corr(method='pearson')
        
        # --- NEW: Add minimum support threshold ---
        # Create a binary matrix (1 if rated, 0 if not)
        binary_matrix = matrix.notna().astype(int)
        
        # Calculate the co-rater count matrix using a dot product
        co_rater_matrix = binary_matrix.T.dot(binary_matrix)
        
        # Set similarity to 0 where the number of co-raters is below the threshold
        similarity_matrix[co_rater_matrix < min_support] = 0
        
        # Fill any remaining NaNs
        similarity_matrix = similarity_matrix.fillna(0)
        
    else:
        raise ValueError("Method must be 'cosine' or 'pearson'")
        
    return similarity_matrix

In [ ]:
def get_item_item_recommendations(movie_id, item_similarity_matrix, item_df, top_n=10):
    if movie_id not in item_similarity_matrix.columns:
        return pd.DataFrame(columns=['movie_id', 'movie_title', 'similarity'])
    sim = item_similarity_matrix[movie_id].drop(movie_id).dropna().sort_values(ascending=False).head(top_n)
    recs = pd.DataFrame({'movie_id': sim.index.astype(int), 'similarity': sim.values})
    recs = recs.merge(item_df[['movie_id','movie_title']], on='movie_id', how='left')
    return recs[['movie_id','movie_title','similarity']]

In [ ]:
item_sim_cosine = prepare_item_similarity(user_item_matrix, method='cosine')
recs_cosine = get_item_item_recommendations(50, item_sim_cosine, item_df)
print("\nRecommendations for 'Star Wars (1977)' using Cosine:\n", recs_cosine)

In [ ]:
# Using Pandas' Pearson Correlation
item_sim_pearson = prepare_item_similarity(user_item_matrix, method='pearson')
recs_pearson = get_item_item_recommendations(50, item_sim_pearson, item_df)
print("\nRecommendations for 'Star Wars (1977)' using Pearson:\n", recs_pearson)

Ranking Improvements
Current method ranks purely on similarity, without considering number of co-ratings.
>     We could weigh similarity scores by the number of users who rated both items to avoid false high similarity from very few co-ratings. --using this handled the error of 1 similarity for all

In [ ]:
# def pearson_recommend(movie_title, min_ratings=50):
#     movie_ratings = user_movie_matrix[movie_title]
#     similar_movies = user_movie_matrix.corrwith(movie_ratings)
#     corr_df = pd.DataFrame(similar_movies, columns=['Correlation'])
#     corr_df = corr_df.join(data.groupby('title')['rating'].count())
#     corr_df.columns = ['Correlation', 'Rating Count']
#     return corr_df[corr_df['Rating Count'] >= min_ratings].sort_values('Correlation', ascending=False)


In [ ]:
# pearson_recommend('Toy Story (1995)').head(10)


## User-User Collaborative Filtering

For User-User similarity, it's beneficial to normalize ratings based on each user's own rating behavior (some users are harsh, others are generous). We do this by subtracting the user's average rating from each of their ratings. This is called row-wise mean-centering.

In [ ]:
def prepare_user_similarity(ratings_matrix, method='cosine', min_support=5):
    # Initialize a consistent variable to return
    similarity_df = pd.DataFrame() 

    if method == 'cosine':
        centered_matrix = ratings_matrix.sub(ratings_matrix.mean(axis=1), axis=0).fillna(0)
        similarity = cosine_similarity(centered_matrix)
        similarity_df = pd.DataFrame(similarity, 
                                     index=ratings_matrix.index, 
                                     columns=ratings_matrix.index)
    elif method == 'pearson':
        similarity_df = ratings_matrix.T.corr(method='pearson').fillna(0)
        # Apply min_support filter if needed (this part is fine)
        counts = ratings_matrix.notna().astype(int) @ ratings_matrix.notna().astype(int).T
        counts = pd.DataFrame(counts, index=ratings_matrix.index, columns=ratings_matrix.index)
        similarity_df[counts < min_support] = 0
    else:
        raise ValueError("method must be 'cosine' or 'pearson'")
    
    return similarity_df

This function will find the top N similar users and recommend movies based on their ratings.

Find Your "Taste Twins": First, the system looks at everyone who has rated movies and finds a small group of people whose rating history is the most similar to yours. These are your "taste twins" – people who tend to love the movies you love and hate the movies you hate.

Give More Weight to Your Best "Twin": It doesn't treat all these twins equally. The opinion of a twin who is extremely similar to you (almost a perfect match) is valued more highly than someone who is only moderately similar.

Create a "Smart" Average Score: The system then looks at all the movies your twins have watched but you haven't. For each of those movies, it calculates a predicted score for you. This score is a weighted average of your twins' ratings, where the ratings from your closest twins have a much bigger impact.

Recommend the Best: Finally, it sorts these predicted scores and recommends the movies with the highest scores.

### User-Based CF with Baseline Estimates

This method improves upon the simple weighted average by accounting for individual user rating biases (some users give higher ratings on average than others). The prediction is calculated by starting with the target user's average rating and adding a weighted average of how other similar users' ratings *deviate* from *their* own averages.

The formula is:

$$ \text{pred}(u, i) = \bar{r}_u + \frac{\sum_{v \in N} s_{u,v} \cdot (r_{v,i} - \bar{r}_v)}{\sum_{v \in N} |s_{u,v}|} $$

**Where:**
* **$\text{pred}(u, i)$**: The predicted rating for user `u` on item `i`.
* **$\bar{r}_u$**: The average rating of the target user `u`.
* **$N$**: The set of neighbor users who have rated item `i`.
* **$s_{u,v}$**: The similarity between user `u` and neighbor `v`.
* **$r_{v,i} - \bar{r}_v$**: The rating of neighbor `v` for item `i`, adjusted by subtracting neighbor `v`'s own average rating. This is the "deviation."

In [ ]:
def get_user_user_recommendations_vectorized(user_id,user_item_matrix, user_sim_df, top_n=10):
    # Mean ratings per user
    user_means = user_item_matrix.mean(axis=1)

    # Movies already rated by the user
    rated_items = user_item_matrix.loc[user_id].dropna().index

    # Unrated movies
    unrated_items = user_item_matrix.columns.difference(rated_items)

    predictions = {}

    for item in unrated_items:
# Users who rated this item
        other_users = user_item_matrix[item].dropna().index

# Similarities between target user and other users
        sims = user_sim_df.loc[user_id, other_users]

# Ratings of other users for this item
        other_ratings = user_item_matrix.loc[other_users, item]

# Deviations from mean ratings
        devs = other_ratings - user_means.loc[other_users]

# Weighted deviation prediction
        numerator = (sims * devs).sum()
        denominator = sims.abs().sum()

        if denominator > 0:
            pred = user_means[user_id] + numerator / denominator
            predictions[item] = pred

    # Sort and return top_n recommendations
    return sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:top_n]


In [ ]:
user_sim_pearson = prepare_user_similarity(user_item_matrix, method='pearson')


# 2. Call your new recommendation function
target_user_id = 196
recommendations = get_user_user_recommendations_vectorized(
    user_id=target_user_id,
    user_item_matrix=user_item_matrix, 
    user_sim_df=user_sim_pearson, 
    top_n=10
)

# 3. Format and print the results
# Convert the list of tuples output to a DataFrame
recs_df = pd.DataFrame(recommendations, columns=['movie_id', 'predicted_rating'])

# Merge with movie titles for a readable output
recs_df = recs_df.merge(item_df[['movie_id', 'movie_title']], on='movie_id', how='left')

print(f"Recommendations for User {target_user_id} using Weighted Deviation (Pearson):\n")
print(recs_df[['movie_id', 'movie_title', 'predicted_rating']])

In [ ]:
# Prepare cosine similarity
user_sim_cosine = prepare_user_similarity(user_item_matrix, method='cosine')

# Get recommendations using cosine similarity
recommendations_cosine = get_user_user_recommendations_vectorized(
    user_id=target_user_id,
    user_item_matrix=user_item_matrix,
    user_sim_df=user_sim_cosine,
    top_n=10
)

# Format and print results
recs_df = pd.DataFrame(recommendations_cosine, columns=['movie_id', 'predicted_rating'])
recs_df = recs_df.merge(item_df[['movie_id', 'movie_title']], on='movie_id', how='left')

print(f"Recommendations for User {target_user_id} using Weighted Deviation (Cosine):\n")
print(recs_df[['movie_id', 'movie_title', 'predicted_rating']])


Why are the Ratings Above 5.0?
The prediction formula you are using is:

Predicted Rating = User's Average Rating + Weighted Average of Neighbor's Deviations

This can result in a score above 5.0. For example, if a user's average rating is 4.2 and their very similar neighbors rated a movie much higher than their own personal averages, the "weighted average of deviations" could be a large positive number like +1.5. The final predicted score would be 4.2 + 1.5 = 5.7, which is outside the 1-5 scale.

In this section, we will **directly compare** Cosine Similarity and Pearson Correlation for our recommendation system.

**Why compare?**
- **Cosine Similarity** measures the angle between rating vectors, ignoring magnitude. Works well when absolute rating values differ but patterns are similar.
- **Pearson Correlation** accounts for mean-centering of ratings, reducing bias from users/movies that rate consistently high or low.

We will:
1. Pick a **sample movie** for Item-Item recommendations.
2. Pick a **sample user** for User-User recommendations.
3. Generate **top-N recommendations** using both similarity methods.
4. Compare the results **side-by-side**.
5. Calculate **Top-K overlap** between the methods for each approach.

The overlap metric will give us a quantitative measure of how similar the results are.  
High overlap → methods agree.  
Low overlap → methods give diverse results (which may or may not be desirable depending on goals).

In [ ]:
# --- 1. Filter the data using your KneeLocator thresholds ---
# (Assuming user_rating_counts and movie_rating_counts are defined from your EDA)
min_user_ratings = user_rating_counts.values[knee_user.knee-1]
min_movie_ratings = movie_rating_counts.values[knee_movie.knee-1]

# Get the IDs of users and movies that meet the threshold
active_users = user_rating_counts[user_rating_counts >= min_user_ratings].index
popular_movies = movie_rating_counts[movie_rating_counts >= min_movie_ratings].index

# Filter the main DataFrame
ratings_df_filtered = df[
    df['user_id'].isin(active_users) & df['movie_id'].isin(popular_movies)
]

print(f"Original ratings: {len(df)}")
print(f"Filtered ratings: {len(ratings_df_filtered)}")

# Model testing

🚨 **The Big Pivot: Overcoming Sparsity**
We just saw how memory-based methods (Item-Item) work. But remember our EDA? **93.7% of our matrix is empty!** Memory-based models struggle here because they need users to have rated the *exact same items* to find similarity. This is where Matrix Factorization saves the day. By compressing the empty grid into dense 'Latent Features', we can find connections between users and items even if they have zero direct overlap.

Why SVD?
Singular Value Decomposition (SVD) is a matrix factorization technique that decomposes the user–item rating matrix into lower-dimensional latent factors. In recommendation systems, these factors capture hidden user preferences and item attributes, enabling better predictions even when data is sparse. SVD is efficient, works well with explicit ratings, and is available in the Surprise library with built-in evaluation functions.

🧠 **Building Intuition: What is a 'Latent Feature'?**
When SVD factorizes the matrix, it creates a lower-dimensional space of 'latent' (hidden) features. The algorithm doesn't know what 'Action' or 'Romance' is. It simply discovers that certain groups of movies are consistently rated similarly by certain groups of people. 
A latent feature could mathematically represent 'Amount of CGI' or 'Darkness of Plot'. By mapping both users and items into this same hidden space, we can predict that a user will like a movie if their latent vectors align, even if they have zero direct co-ratings with similar users.

In [ ]:
# FIX: Sort by timestamp for temporal split to avoid time-travel data leakage
ratings_df_filtered = ratings_df_filtered.sort_values('timestamp')

# Split 80/20 using Pandas
split_idx = int(len(ratings_df_filtered) * 0.8)
train_df = ratings_df_filtered.iloc[:split_idx]
test_df = ratings_df_filtered.iloc[split_idx:]

reader = Reader(rating_scale=(1, 5))

# Create Surprise datasets (isolated train/test)
data_train = Dataset.load_from_df(train_df[['user_id', 'movie_id', 'rating']], reader)
data_test = Dataset.load_from_df(test_df[['user_id', 'movie_id', 'rating']], reader)

# Build full trainset for the initial algorithm training
trainset = data_train.build_full_trainset()
# Testset format in Surprise is a list of tuples: (user, item, rating)
testset = [tuple(x) for x in test_df[['user_id', 'movie_id', 'rating']].values]


In [ ]:
algo = SVD(random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)

In [ ]:
print(f"The default number of latent features (n_factors) is: {algo.n_factors}")

In [ ]:
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

Evaluation Results

RMSE: 0.8535

MAE: 0.6575

Interpretation:
RMSE (Root Mean Square Error) measures the average magnitude of prediction errors, giving more weight to larger errors. MAE (Mean Absolute Error) measures the average absolute difference between predicted and actual ratings.
For this dataset, an RMSE of ~0.85 indicates the model’s predictions are fairly close to actual ratings (on a 1–5 scale). MAE of ~0.66 suggests, on average, predictions deviate from actual ratings by less than 1 rating point.

While this is a good baseline, further improvements can be made through hyperparameter tuning and hybrid approaches.

### Precision@K 

**Precision@K** measures how many of the **top-K recommended items** are actually **relevant** to the user.  
It’s a basic but important evaluation metric for recommender systems.

**Formula:** Precision@K = (# of relevant items in top-K) / K

**Why are we using it now?**
- We've just generated recommendations with different similarity measures (Cosine, Pearson).
- We need an **objective way** to measure which one better matches user preferences.

**Where will we use it later?**
- **Offline evaluation** before deploying models (e.g., SVD, hybrid models).
- **Comparing multiple algorithms** fairly.
- **Hyperparameter tuning** (e.g., choosing K in KNN).

Since we have an **explicit ratings dataset**, we can define "relevant" as:
> Movies the user rated **≥ 4.0** (or another threshold).


In [ ]:
def precision_recall_at_k(predictions, k=10, threshold=4.0):
    # First map the predictions to each user.
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    for uid, user_ratings in user_est_true.items():
# Sort user ratings by estimated value
        user_ratings.sort(key=lambda x: x[0], reverse=True)

# Number of relevant items
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)

# Number of recommended items in top k
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])

# Number of relevant and recommended items in top k
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

# Precision@k: Proportion of recommended items that are relevant
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 1

# Recall@k: Proportion of relevant items that are recommended
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 1

    # Average the values across all users
    avg_precision = sum(prec for prec in precisions.values()) / len(precisions)
    avg_recall = sum(rec for rec in recalls.values()) / len(recalls)
    
    return avg_precision, avg_recall

In [ ]:
precision, recall = precision_recall_at_k(predictions, k=10, threshold=4.0)
print(f"Precision@10: {precision:.4f}")
print(f"Recall@10: {recall:.4f}")

What it means: This is an excellent score. It means that when your model recommends a list of 10 movies to a user, on average, 92.7% of those recommendations are relevant (i.e., movies that the user actually rated 4 stars or higher).

In Layman's Terms: This is the "quality" metric. It tells you that the recommendations your model puts at the top of the list are highly likely to be good ones. There is very little "junk" in your top 10.

What it means: This is a solid result. It means that your top 10 list is successfully finding and showing the user about 57% of all the "relevant" movies that were available for them in the test set.

In Layman's Terms: This is the "coverage" metric. It tells you how many of the movies the user would have liked you were able to find. A perfect recall would mean your top 10 list contained every single movie the user liked.

The Precision-Recall Trade-off:
There's often a trade-off. You could increase recall by recommending more movies (e.g., a top 50 list), but this would likely lower your precision because a longer list has a higher chance of containing irrelevant movies. Your model currently has very high precision, which is often the most important factor for a good user experience.

## Hyperparameter Tuning for SVD 

We first ran a baseline SVD model, which gave us an **RMSE of 0.8535**.

From prior literature on MovieLens datasets and our own baseline results, we can make the following observations:

### n_factors
- Baseline was using Surprise's default (`n_factors=100`)
- Values too low (<70) underfit, while too high (>130) slightly risk overfitting with limited gains
- **We will explore a tight band: [80, 100, 120]**

### n_epochs
- RMSE improved with higher epochs up to ~50 in prior experiments
- Past 60, improvements diminish
- **Narrowed range: [40, 50, 60]**

### lr_all
- Learning rates >0.007 can destabilize training
- 0.004–0.006 worked well in smaller tests
- **Narrowed range: [0.004, 0.005, 0.006]**

### reg_all
- Lower regularization (<0.03) risks overfitting, especially with higher factors
- Too high (>0.08) harms flexibility
- **Narrowed range: [0.03, 0.05, 0.07]**

---

By narrowing the range, we reduce the search space from **81 combinations** (3×3×3×3) to **27 combinations**, cutting runtime while staying in the most promising region.

In [ ]:
param_grid = {
    'n_factors': [60, 80, 100],      
    'n_epochs': [50, 60, 70],         
    'lr_all': [0.005, 0.006, 0.007], 
    'reg_all': [0.05, 0.07, 0.09]     
}

# Grid Search
gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse', 'mae'],
    cv=5,
    n_jobs=-1,
    joblib_verbose=2
)
# FIX: Fit only on training data to prevent test set leakage into CV folds
gs.fit(data_train)

# Best parameters and score
print("Best RMSE Score:", gs.best_score['rmse'])
print("Best Params:", gs.best_params['rmse'])

# Retrain the model using the best parameters
best_algo = SVD(**gs.best_params['rmse'])
best_algo.fit(trainset)

In [ ]:
print(f"The default number of latent features (n_factors) is: {best_algo.n_factors}")

In [ ]:
# Baseline model (default params)
baseline_model = SVD(random_state=42)
baseline_model.fit(trainset)
baseline_predictions = baseline_model.test(testset)
baseline_rmse = accuracy.rmse(baseline_predictions, verbose=False)

# Tuned model (best params from GridSearchCV)
tuned_model = SVD(**gs.best_params['rmse'], random_state=42)
tuned_model.fit(trainset)
tuned_predictions = tuned_model.test(testset)
tuned_rmse = accuracy.rmse(tuned_predictions, verbose=False)

print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"Tuned RMSE:    {tuned_rmse:.4f}")
print(f"Improvement:   {baseline_rmse - tuned_rmse:.4f}")

### Visualizing the Prediction Errors
Looking at a single RMSE number is good, but plotting the actual error distribution helps us see *how* the model is failing. Are we consistently over-predicting? Under-predicting? Let's check.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the error for each prediction (Predicted - Actual)
errors = [pred.est - pred.r_ui for pred in baseline_predictions]

plt.figure(figsize=(10, 6))
sns.histplot(errors, bins=50, kde=True, color='purple')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (Zero Error)')
plt.title('Distribution of Prediction Errors (SVD Baseline)')
plt.xlabel('Error in Stars (Predicted - Actual)')
plt.ylabel('Frequency')
plt.legend()
plt.show()


Why a Paired t-Test?
We’re testing two algorithms on the same test set.

Predictions are paired per user–item combination.

A paired t-test on RMSE/MAE per fold (or per sample errors) can tell us if the observed difference is due to chance.

In [ ]:
from scipy.stats import ttest_rel

# Get prediction errors
baseline_errors = np.array([true_r - est for (_, _, true_r, est, _) in baseline_predictions])
tuned_errors    = np.array([true_r - est for (_, _, true_r, est, _) in tuned_predictions])

# Paired t-test
t_stat, p_value = ttest_rel(baseline_errors, tuned_errors)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("✅ Statistically significant difference.")
else:
    print("❌ No statistically significant difference.")


T-statistic: 1.2190

P-value: 0.2234

Conclusion: p > 0.05 ⇒ No statistically significant difference.
This means the observed RMSE change is likely due to random variation rather than a true improvement.

Parameter tuning in this range did not yield a statistically significant improvement over the baseline SVD model. We therefore chose to retain the baseline configuration as our reference point for further analysis."

## Cold Start Problem and Handling

### What is the Cold Start Problem?
In recommender systems, the **cold start problem** occurs when we have to make recommendations for:
- **New users**: No prior ratings, so collaborative filtering can’t estimate their preferences.
- **New items**: No ratings from any users, so the system cannot estimate relevance.
- **New system**: When the dataset is very sparse at the beginning.

Collaborative filtering methods like **User-User CF** or **SVD** depend heavily on historical rating data.  
If that data is missing for a user/item, the model can’t make personalized predictions.

---

We will implement a **popularity-based fallback**:
- **For new users**: Recommend the most popular movies based on the number of ratings (and optionally their average rating).
- **For new items**: Recommend them only after they reach a minimum number of ratings; until then, they won’t appear in personalized recommendations.

This ensures our system always returns recommendations, even without prior user history.


⚠️ **The #1 Business Risk: Cold Start Abandonment**
If a new user signs up and sees an empty dashboard or terrible random recommendations, they will churn immediately. The Cold Start problem isn't just an ML annoyance; it's a critical user retention issue. 
Falling back to highly-rated, highly-popular movies (Bayesian weighted) is the industry standard 'safe bet' until enough implicit (clicks) or explicit (ratings) data is gathered to switch to personalized SVD recommendations.

## Final Recommendations with Cold-Start Handling

Now that we have identified our best-performing model (baseline SVD), we retrain it on the full dataset to maximize available data. 
We then implement a final recommendation function that:
- Generates personalized predictions for known users.
- Falls back to most popular movies for cold-start (new) users.


In [ ]:
# FIX: Recompute popularity score strictly on training data to prevent target leakage
rating_stats_train = train_df.groupby('movie_id')['rating'].agg(['mean','count']).reset_index()
rating_stats_train.columns = ['movie_id', 'avg_rating', 'num_rating']

C_train = (rating_stats_train['avg_rating'] * rating_stats_train['num_rating']).sum() / rating_stats_train['num_rating'].sum()
m_train = int(rating_stats_train['num_rating'].quantile(0.25))
m_train = max(1, m_train)

rating_stats_train['popularity_score'] = (
    (rating_stats_train['num_rating'] / (rating_stats_train['num_rating'] + m_train)) * rating_stats_train['avg_rating'] +
    (m_train / (rating_stats_train['num_rating'] + m_train)) * C_train
)

popular_movies_train_df = rating_stats_train.merge(item_df[['movie_id', 'movie_title']], on='movie_id', how='left')

def get_final_recommendations(user_id, algo, trainset=None, item_df=item_df, popular_movies_df=popular_movies_train_df, n=10):
    # Use the trainset attached to algo if not explicitly provided
    if trainset is None:
        trainset = getattr(algo, 'trainset', None)
        if trainset is None:
            raise ValueError("trainset not provided and algo is not fitted (no algo.trainset).")

    try:
        # check whether user exists in trainset
        inner_user = trainset.to_inner_uid(user_id)

        # all raw item ids in trainset
        all_movie_ids = [trainset.to_raw_iid(iid) for iid in trainset.all_items()]

        # movies already rated by user
        rated_movie_ids = [trainset.to_raw_iid(iid) for iid, _ in trainset.ur[inner_user]]

        # build list of unrated raw ids
        unrated = [mid for mid in all_movie_ids if mid not in rated_movie_ids]

        # predict for unrated items
        preds = [algo.predict(user_id, mid) for mid in unrated]

        # pick top n
        preds = sorted(preds, key=lambda p: p.est, reverse=True)[:n]

        # build DataFrame (movie_id as int, predicted_rating clipped & rounded)
        recs = pd.DataFrame([(int(p.iid), p.est) for p in preds], columns=['movie_id','predicted_rating'])
        recs['predicted_rating'] = recs['predicted_rating'].clip(1, 5).round(2)

        # merge titles
        recs = recs.merge(item_df[['movie_id','movie_title']], on='movie_id', how='left')
        return recs[['movie_id','movie_title','predicted_rating']].sort_values('predicted_rating', ascending=False)

    except ValueError:
    # Cold-start case — user not found in training data
        recommendations_df = popular_movies_df.copy()
        recommendations_df['predicted_rating'] = recommendations_df['avg_rating']
        recommendations_df = recommendations_df.sort_values('predicted_rating', ascending=False)
        return recommendations_df[['movie_id', 'movie_title', 'predicted_rating']].head(n)


In [ ]:
# Train on full dataset
# Build full trainset on ALL available data for final production model
full_data = Dataset.load_from_df(ratings_df_surprise, reader)
full_trainset = full_data.build_full_trainset()
algo.fit(full_trainset)

print("Known user example:")
display(get_final_recommendations(
    222, algo, full_trainset, item_df, popular_movies_df, n=10
))

print("Cold-start example:")
display(get_final_recommendations(
    9999, algo, full_trainset, item_df, popular_movies_df, n=10
))


---
## 🎬 Wrapping It Up: The Final Review

Alright, let's step back and look at the big picture of what we just built. What a journey!

### What we did:
We started with a massive, mostly empty grid of movie ratings (over 93% empty!). We cleaned it up, explored who our power-users were, and calculated a smart **Bayesian popularity score** so that weird, obscure movies with one 5-star rating didn't break our system. Then we built actual recommendation engines: first using old-school memory-based methods (finding similar users/items), and finally pulling out the heavy artillery with **SVD (Singular Value Decomposition)** to find hidden patterns in the chaos.

### Why we did it:
Because building a recommender system isn't just about calling `model.fit()`. The memory-based methods (Cosine, Pearson) are super intuitive, but they fall apart when data is extremely sparse. SVD handles the sparsity beautifully by compressing the users and movies into 'latent features'. We also built that popularity fallback because of the **Cold Start problem**—if a brand new user joins, we literally know nothing about them, but we still need to show them *something* good so they don't bounce off the app.

### What we saw:
Our baseline SVD model ended up being the MVP. We tried tuning the hyperparameters with GridSearch, but the stats (paired t-test) told us the tiny improvement wasn't actually mathematically significant. We achieved an **RMSE of ~0.85**, which means our predictions are generally off by less than a single star. More importantly, our **Precision@10 was over 92%!** That means if we recommend 10 movies to a user, 9 of them are likely to be hits (rated 4.0+).

Building this pipeline taught me that dealing with data leakage (like making sure we do strict time-based splitting) and handling edge cases (like the cold start) is where the real engineering happens. It's not just about math; it's about building a robust product.